# LIAR-PLUS — Exploratory Data Analysis
### Thesis: Explainable Fact-Checking via LLM-Generated Synthetic Justifications
*Gianluigi Bianco — Chapter 4: Synthetic Justification Generation*

This notebook performs a comprehensive EDA of the LIAR-PLUS dataset across train, validation and test splits.
The goal is to extract structural and linguistic insights about **claims** and **justifications** that will
directly inform the prompt design phase (Chapter 4.3).

---


## 1. Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import re, collections, string, os, warnings

warnings.filterwarnings('ignore')

# ── Style ──────────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 130
plt.rcParams['axes.spines.top']  = False
plt.rcParams['axes.spines.right'] = False

LABEL_ORDER  = ['true', 'mostly-true', 'half-true', 'barely-true', 'false', 'pants-fire']
LABEL_COLORS = ['#2ecc71','#82e0aa','#f7dc6f','#f0a500','#e74c3c','#900c3f']
LABEL_PALETTE = dict(zip(LABEL_ORDER, LABEL_COLORS))

PARTY_COLORS = {'democrat':'#3498db','republican':'#e74c3c',
                'none':'#95a5a6','independent':'#9b59b6',
                'organization':'#e67e22','journalist':'#1abc9c'}

OUTPUT_DIR = 'eda_figures'
os.makedirs(OUTPUT_DIR, exist_ok=True)

def savefig(name):
    path = f'{OUTPUT_DIR}/{name}.png'
    plt.savefig(path, bbox_inches='tight', dpi=150)
    plt.show()
    print(f'Saved → {path}')

print('Setup complete.')


## 2. Load & Merge Splits

In [ ]:
COLS = ['id', 'json_id', 'label', 'claim', 'subject', 'speaker',
        'speaker_job', 'state', 'party',
        'barely_true_cnt', 'false_cnt', 'half_true_cnt',
        'mostly_true_cnt', 'pants_fire_cnt',
        'venue', 'justification']

splits = {}
for split, path in [('train', 'train2.tsv'),
                    ('val',   'val2.tsv'),
                    ('test',  'test2.tsv')]:
    df_s = pd.read_csv(path, sep='\t', header=None, names=COLS,
                       quoting=3, on_bad_lines='skip')
    df_s['split'] = split
    splits[split] = df_s
    print(f'{split:5s}: {len(df_s):>5,} rows')

df = pd.concat(splits.values(), ignore_index=True)

# ── Basic cleaning ──────────────────────────────────────────────────────────
df['label']    = df['label'].str.strip().str.lower()
df['party']    = df['party'].fillna('none').str.strip().str.lower()
df['speaker']  = df['speaker'].fillna('unknown').str.strip()
df['claim']    = df['claim'].fillna('').astype(str)
df['justification'] = df['justification'].fillna('').astype(str)
df['venue']    = df['venue'].fillna('unknown').str.strip()
df['state']    = df['state'].fillna('unknown').str.strip()

# Normalise 'pants-on-fire' → 'pants-fire' 
df['label'] = df['label'].replace({'pants-on-fire': 'pants-fire'})

CREDIT_COLS = ['barely_true_cnt','false_cnt','half_true_cnt',
               'mostly_true_cnt','pants_fire_cnt']
for c in CREDIT_COLS:
    df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0)
df['total_statements'] = df[CREDIT_COLS].sum(axis=1)

print(f'\nMerged dataset: {len(df):,} rows × {len(df.columns)} columns')
print(f'Splits: {df.groupby("split").size().to_dict()}')
print(f'Labels: {df["label"].value_counts().to_dict()}')
print(f'Missing justifications: {(df["justification"]=="").sum():,}')


## 3. Label Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall distribution
vc = df['label'].value_counts().reindex(LABEL_ORDER, fill_value=0)
bars = axes[0].bar(vc.index, vc.values,
                   color=[LABEL_PALETTE.get(l,'#999') for l in vc.index],
                   edgecolor='white', linewidth=0.7)
for bar, val in zip(bars, vc.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
                 f'{val:,}\n({val/len(df)*100:.1f}%)',
                 ha='center', va='bottom', fontsize=9)
axes[0].set_title('Label Distribution — Full Dataset', fontweight='bold')
axes[0].set_xlabel('Label'); axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=15)

# Per split
split_label = (df.groupby(['split','label'])
                 .size().unstack(fill_value=0)
                 .reindex(columns=LABEL_ORDER, fill_value=0)
                 .reindex(['train','val','test']))
split_label_pct = split_label.div(split_label.sum(axis=1), axis=0) * 100
split_label_pct.plot(kind='bar', ax=axes[1], color=LABEL_COLORS,
                     edgecolor='white', linewidth=0.5)
axes[1].set_title('Label Distribution per Split (% of split)', fontweight='bold')
axes[1].set_xlabel('Split'); axes[1].set_ylabel('Percentage')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(loc='upper right', fontsize=8, ncol=2)

plt.tight_layout()
savefig('01_label_distribution')

# Summary table
print("\n─── Label counts per split ───")
print(split_label.to_string())


## 4. Claim Text Analysis

In [ ]:
# ── Token & character length ─────────────────────────────────────────────
df['claim_tokens'] = df['claim'].apply(lambda x: len(x.split()))
df['claim_chars']  = df['claim'].apply(len)

print("─── Claim length statistics (tokens) ───")
print(df['claim_tokens'].describe().round(1).to_string())
print()
print("─── Claim length statistics (characters) ───")
print(df['claim_chars'].describe().round(1).to_string())


In [ ]:
# ── Distribution of claim length by label ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of token counts
axes[0].hist(df['claim_tokens'], bins=50, color='#3498db', alpha=0.8, edgecolor='white')
axes[0].axvline(df['claim_tokens'].median(), color='#e74c3c', linestyle='--',
                label=f'Median = {df["claim_tokens"].median():.0f}')
axes[0].axvline(df['claim_tokens'].mean(), color='#f39c12', linestyle='-.',
                label=f'Mean = {df["claim_tokens"].mean():.1f}')
axes[0].set_title('Claim Length Distribution (tokens)', fontweight='bold')
axes[0].set_xlabel('Tokens'); axes[0].set_ylabel('Count')
axes[0].legend()

# Boxplot by label
label_order_present = [l for l in LABEL_ORDER if l in df['label'].unique()]
df_box = df[df['label'].isin(label_order_present)]
sns.boxplot(data=df_box, x='label', y='claim_tokens', order=label_order_present,
            palette=LABEL_PALETTE, ax=axes[1], linewidth=0.8)
axes[1].set_title('Claim Length by Label', fontweight='bold')
axes[1].set_xlabel('Label'); axes[1].set_ylabel('Tokens')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
savefig('02_claim_length')


In [ ]:
# ── Structural patterns in claims ─────────────────────────────────────────
def has_quote(text):
    return bool(re.search(r'[\'\"\u201c\u201d\u2018\u2019]', text))

def has_number(text):
    return bool(re.search(r'\b\d+[\.,]?\d*\s*(%|percent|million|billion|trillion|thousand)?', text, re.I))

def has_proper_noun(text):
    # Heuristic: capitalised word not at sentence start (after first word)
    words = text.split()
    return any(w[0].isupper() for w in words[1:] if w.isalpha())

def has_comparative(text):
    return bool(re.search(r'\b(more|less|fewer|greater|higher|lower|most|least|best|worst|largest|smallest)\b', text, re.I))

def has_temporal(text):
    return bool(re.search(r'\b(since|in \d{4}|last year|this year|over the past|\d{4}|today|now|recently|always|never)\b', text, re.I))

def has_attribution(text):
    return bool(re.search(r'\b(says?|said|claims?|according to|stated|argued)\b', text, re.I))

patterns = {
    'Contains quote marks'    : 'has_quote',
    'Contains number/stat'    : 'has_number',
    'Contains proper noun'    : 'has_proper_noun',
    'Comparative language'    : 'has_comparative',
    'Temporal reference'      : 'has_temporal',
    'Attribution phrase'      : 'has_attribution',
}
fns = {
    'has_quote'        : has_quote,
    'has_number'       : has_number,
    'has_proper_noun'  : has_proper_noun,
    'has_comparative'  : has_comparative,
    'has_temporal'     : has_temporal,
    'has_attribution'  : has_attribution,
}

for label, fn_name in patterns.items():
    df[fn_name] = df['claim'].apply(fns[fn_name])

pattern_counts = {label: df[fn].sum() for label, fn in patterns.items()}

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(list(pattern_counts.keys()),
               [v/len(df)*100 for v in pattern_counts.values()],
               color='#5b8def', edgecolor='white')
for bar, (lbl, val) in zip(bars, pattern_counts.items()):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:,} ({val/len(df)*100:.1f}%)', va='center', fontsize=9)
ax.set_xlabel('% of Claims')
ax.set_title('Structural Patterns in Claims', fontweight='bold')
ax.set_xlim(0, 105)
plt.tight_layout()
savefig('03_claim_patterns')

print("\n─── Pattern counts ───")
for lbl, val in pattern_counts.items():
    print(f'{lbl:<30} {val:>5,}  ({val/len(df)*100:.1f}%)')


In [ ]:
# ── Pattern presence by label ──────────────────────────────────────────────
pattern_cols = list(fns.keys())
pattern_by_label = (df.groupby('label')[pattern_cols]
                      .mean() * 100
                      .reindex(LABEL_ORDER))

fig, ax = plt.subplots(figsize=(12, 5))
pattern_by_label.T.plot(kind='bar', ax=ax,
    color=[LABEL_COLORS[i] for i in range(len(LABEL_ORDER))],
    edgecolor='white', linewidth=0.5)
ax.set_title('Claim Pattern Prevalence by Label (%)', fontweight='bold')
ax.set_xlabel('Pattern'); ax.set_ylabel('% of claims with pattern')
ax.tick_params(axis='x', rotation=20)
ax.legend(title='Label', fontsize=8, bbox_to_anchor=(1.01,1), loc='upper left')
short_labels = ['Quote','Number','Proper\nNoun','Compar.','Temporal','Attribution']
ax.set_xticklabels(short_labels)
plt.tight_layout()
savefig('04_claim_patterns_by_label')


## 5. Justification Analysis

In [ ]:
# ── Filter out empty justifications ───────────────────────────────────────
df_j = df[df['justification'].str.strip() != ''].copy()
print(f'Rows with justification: {len(df_j):,} / {len(df):,}  ({len(df_j)/len(df)*100:.1f}%)')

df_j['just_tokens'] = df_j['justification'].apply(lambda x: len(x.split()))
df_j['just_chars']  = df_j['justification'].apply(len)

# Also add to full df for later joins
df['just_tokens'] = df['justification'].apply(lambda x: len(x.split()) if x.strip() else np.nan)
df['just_chars']  = df['justification'].apply(lambda x: len(x) if x.strip() else np.nan)

print("\n─── Justification length statistics (tokens) ───")
print(df_j['just_tokens'].describe().round(1).to_string())


In [ ]:
# ── Length distributions: claim vs justification ───────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_j['just_tokens'], bins=60, color='#8e44ad', alpha=0.8, edgecolor='white')
axes[0].axvline(df_j['just_tokens'].median(), color='#e74c3c', linestyle='--',
                label=f'Median = {df_j["just_tokens"].median():.0f}')
axes[0].axvline(df_j['just_tokens'].mean(), color='#f39c12', linestyle='-.',
                label=f'Mean = {df_j["just_tokens"].mean():.1f}')
axes[0].set_title('Justification Length (tokens)', fontweight='bold')
axes[0].set_xlabel('Tokens'); axes[0].set_ylabel('Count')
axes[0].legend()

# Side-by-side boxplot of claim vs justification length per label
df_long = pd.concat([
    df_j[['label','claim_tokens']].rename(columns={'claim_tokens':'tokens'}).assign(type='Claim'),
    df_j[['label','just_tokens']].rename(columns={'just_tokens':'tokens'}).assign(type='Justification')
])
sns.boxplot(data=df_long[df_long['label'].isin(LABEL_ORDER)],
            x='label', y='tokens', hue='type',
            order=LABEL_ORDER, palette={'Claim':'#5b8def','Justification':'#8e44ad'},
            ax=axes[1], linewidth=0.8)
axes[1].set_title('Claim vs Justification Length by Label', fontweight='bold')
axes[1].set_xlabel('Label'); axes[1].set_ylabel('Tokens')
axes[1].tick_params(axis='x', rotation=15)
axes[1].legend(title='Type')

plt.tight_layout()
savefig('05_justification_length')


In [ ]:
# ── Opening phrases (first 4 words) ─────────────────────────────────────────
df_j['just_open4'] = df_j['justification'].apply(
    lambda x: ' '.join(x.strip().split()[:4]).lower()
              .translate(str.maketrans('','', string.punctuation))
)
top_opens = collections.Counter(df_j['just_open4']).most_common(20)

fig, ax = plt.subplots(figsize=(11, 6))
phrases, counts = zip(*top_opens)
ax.barh(list(reversed(phrases)), list(reversed(counts)),
        color='#27ae60', edgecolor='white')
ax.set_title('Top 20 Justification Opening Phrases (first 4 words)', fontweight='bold')
ax.set_xlabel('Frequency')
plt.tight_layout()
savefig('06_justification_openings')


In [ ]:
# ── Most frequent content words in justifications ─────────────────────────
STOPWORDS = set("""a an the and or but in on at to of for is are was were be been
    being have has had do does did will would could should may might shall
    this that these those it its with from by about as into through during
    also just very so then when where which who whom what while how all any
    both each few more most other some such no nor not only same so than too
    very s t can will just don should now i me my we our you your he she
    his her its they them their what which who this that am is are was were""".split())

CUSTOM_STOP = {'say','said','says','would','could','also','one','two',
               'year','years','people','claim','claims','statement','however',
               'although','therefore','thus','since','even','still','already',
               'like','get','got','make','made','much','many','well','new'}

def top_words(series, n=40):
    counter = collections.Counter()
    for text in series:
        words = re.findall(r'[a-z]+', text.lower())
        counter.update(w for w in words if w not in STOPWORDS
                       and w not in CUSTOM_STOP and len(w) > 2)
    return counter.most_common(n)

top_j = top_words(df_j['justification'])
top_c = top_words(df['claim'])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, data, title, color in [
        (axes[0], top_c[:30], 'Top 30 Words — Claims', '#3498db'),
        (axes[1], top_j[:30], 'Top 30 Words — Justifications', '#8e44ad')]:
    words, counts = zip(*data)
    ax.barh(list(reversed(words)), list(reversed(counts)),
            color=color, alpha=0.85, edgecolor='white')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Frequency')
plt.tight_layout()
savefig('07_top_words')


## 6. Speaker, Party & State Analysis

In [ ]:
# ── Party distribution ─────────────────────────────────────────────────────
party_vc  = df['party'].value_counts().head(8)
party_lbl_pct = (df[df['party'].isin(party_vc.index)]
                   .groupby(['party','label']).size()
                   .unstack(fill_value=0)
                   .reindex(columns=LABEL_ORDER, fill_value=0))
party_lbl_pct = party_lbl_pct.div(party_lbl_pct.sum(axis=1), axis=0)*100

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].bar(party_vc.index, party_vc.values,
            color=[PARTY_COLORS.get(p,'#bdc3c7') for p in party_vc.index],
            edgecolor='white')
axes[0].set_title('Statements per Party', fontweight='bold')
axes[0].set_xlabel('Party'); axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=20)

party_lbl_pct.reindex(['democrat','republican','none','independent'],
                       fill_value=0).plot(
    kind='bar', ax=axes[1], color=LABEL_COLORS, edgecolor='white', linewidth=0.5)
axes[1].set_title('Label Distribution by Party (%)', fontweight='bold')
axes[1].set_xlabel('Party'); axes[1].set_ylabel('% within party')
axes[1].tick_params(axis='x', rotation=15)
axes[1].legend(fontsize=8, ncol=2)

plt.tight_layout()
savefig('08_party_analysis')


In [ ]:
# ── Top speakers ───────────────────────────────────────────────────────────
top_speakers = df['speaker'].value_counts().head(15)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].barh(list(reversed(top_speakers.index)),
             list(reversed(top_speakers.values)),
             color='#2980b9', edgecolor='white')
axes[0].set_title('Top 15 Speakers by Statement Count', fontweight='bold')
axes[0].set_xlabel('Number of Statements')

# Veracity profile for top 10 speakers
top10 = top_speakers.index[:10].tolist()
spk_lbl = (df[df['speaker'].isin(top10)]
             .groupby(['speaker','label']).size()
             .unstack(fill_value=0)
             .reindex(columns=LABEL_ORDER, fill_value=0))
spk_lbl_pct = spk_lbl.div(spk_lbl.sum(axis=1), axis=0)*100
spk_lbl_pct.plot(kind='barh', ax=axes[1], color=LABEL_COLORS,
                 edgecolor='white', linewidth=0.4, stacked=True)
axes[1].set_title('Veracity Profile — Top 10 Speakers (%)', fontweight='bold')
axes[1].set_xlabel('% of statements')
axes[1].legend(fontsize=8, bbox_to_anchor=(1.01,1), loc='upper left')

plt.tight_layout()
savefig('09_speaker_analysis')


In [ ]:
# ── Top subjects ───────────────────────────────────────────────────────────
subject_counter = collections.Counter()
for subjects in df['subject'].dropna():
    for s in str(subjects).split(','):
        s = s.strip().lower()
        if s:
            subject_counter[s] += 1
top_subjects = subject_counter.most_common(20)
subj, counts = zip(*top_subjects)

fig, ax = plt.subplots(figsize=(11, 7))
ax.barh(list(reversed(subj)), list(reversed(counts)),
        color='#16a085', edgecolor='white')
ax.set_title('Top 20 Subjects / Topics', fontweight='bold')
ax.set_xlabel('Number of Claims')
plt.tight_layout()
savefig('10_subjects')


## 7. Speaker Credit History Analysis

In [ ]:
# ── Distribution of total statements per speaker ──────────────────────────
print("─── Credit history statistics ───")
print(df[CREDIT_COLS + ['total_statements']].describe().round(1).to_string())


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

labels_hist = ['Barely True', 'False', 'Half True', 'Mostly True', 'Pants-Fire', 'Total']
colors_hist = ['#f0a500','#e74c3c','#f7dc6f','#82e0aa','#900c3f','#3498db']

for ax, col, lbl, col_c in zip(axes, CREDIT_COLS + ['total_statements'],
                                 labels_hist, colors_hist):
    data = df[col][df[col] > 0]
    ax.hist(data, bins=40, color=col_c, alpha=0.85, edgecolor='white')
    ax.axvline(data.median(), color='black', linestyle='--', linewidth=1,
               label=f'Median={data.median():.0f}')
    ax.set_title(f'{lbl} Count History', fontweight='bold', fontsize=10)
    ax.set_xlabel('Count'); ax.set_ylabel('Frequency')
    ax.legend(fontsize=8)

plt.suptitle('Speaker Credit History Distributions (non-zero values)', 
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
savefig('11_credit_history')


In [ ]:
# ── Are experienced speakers more truthful? ────────────────────────────────
# Compute truthfulness ratio = (true + mostly_true) / total
df['truth_ratio'] = (df['mostly_true_cnt'] + 
                     df.get('true_cnt', 0)) / (df['total_statements'] + 1e-9)

exp_bins = [0, 5, 20, 50, 200, df['total_statements'].max()+1]
exp_labels = ['1–5','6–20','21–50','51–200','201+']
df['experience'] = pd.cut(df['total_statements'], bins=exp_bins,
                           labels=exp_labels, right=True)

exp_label_pct = (df.groupby(['experience','label'], observed=True)
                   .size().unstack(fill_value=0)
                   .reindex(columns=LABEL_ORDER, fill_value=0))
exp_label_pct_n = exp_label_pct.div(exp_label_pct.sum(axis=1), axis=0)*100

fig, ax = plt.subplots(figsize=(11, 5))
exp_label_pct_n.plot(kind='bar', ax=ax, color=LABEL_COLORS,
                      edgecolor='white', linewidth=0.4)
ax.set_title('Label Distribution by Speaker Experience (total past statements)', fontweight='bold')
ax.set_xlabel('Total past statements'); ax.set_ylabel('% within experience band')
ax.tick_params(axis='x', rotation=0)
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
savefig('12_experience_vs_label')


## 8. Venue and State Distribution

In [ ]:
# ── Top venues ─────────────────────────────────────────────────────────────
venue_vc = df[df['venue'] != 'unknown']['venue'].value_counts().head(20)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].barh(list(reversed(venue_vc.index)), list(reversed(venue_vc.values)),
             color='#d35400', edgecolor='white')
axes[0].set_title('Top 20 Venues / Statement Contexts', fontweight='bold')
axes[0].set_xlabel('Count')

# Top 15 states
state_vc = df[~df['state'].isin(['unknown','none',''])]['state'].value_counts().head(15)
axes[1].bar(state_vc.index, state_vc.values,
            color='#2c3e50', edgecolor='white', alpha=0.85)
axes[1].set_title('Top 15 States by Statement Count', fontweight='bold')
axes[1].set_xlabel('State'); axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
savefig('13_venue_state')


## 9. Claim–Justification Length Ratio & Correlation

In [ ]:
df_cj = df_j.copy()
df_cj['len_ratio'] = df_cj['just_tokens'] / df_cj['claim_tokens'].replace(0, np.nan)

print("─── Justification / Claim token ratio ───")
print(df_cj['len_ratio'].describe().round(2).to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(df_cj['claim_tokens'], df_cj['just_tokens'],
                alpha=0.08, s=8, color='#2980b9')
axes[0].set_xlabel('Claim tokens'); axes[0].set_ylabel('Justification tokens')
axes[0].set_title('Claim vs Justification Length (scatter)', fontweight='bold')
corr = df_cj[['claim_tokens','just_tokens']].corr().iloc[0,1]
axes[0].text(0.05, 0.92, f'Pearson r = {corr:.3f}',
             transform=axes[0].transAxes, fontsize=10)

# Ratio distribution by label
sns.boxplot(data=df_cj[df_cj['label'].isin(LABEL_ORDER)],
            x='label', y='len_ratio', order=LABEL_ORDER,
            palette=LABEL_PALETTE, ax=axes[1], linewidth=0.8)
axes[1].set_title('Justification/Claim Ratio by Label', fontweight='bold')
axes[1].set_xlabel('Label'); axes[1].set_ylabel('Ratio')
axes[1].tick_params(axis='x', rotation=15)
axes[1].set_ylim(0, 50)

plt.tight_layout()
savefig('14_ratio_correlation')


## 10. Summary Statistics Table

In [ ]:
summary = pd.DataFrame({
    'Total statements'        : [len(df)],
    'Train / Val / Test'      : [f'{splits["train"].shape[0]:,} / {splits["val"].shape[0]:,} / {splits["test"].shape[0]:,}'],
    'Unique speakers'         : [df['speaker'].nunique()],
    'Unique subjects'         : [df['subject'].str.split(',').explode().str.strip().nunique()],
    'Claims with justification': [f'{len(df_j):,} ({len(df_j)/len(df)*100:.1f}%)'],
    'Avg claim tokens'        : [f'{df["claim_tokens"].mean():.1f}'],
    'Median claim tokens'     : [f'{df["claim_tokens"].median():.0f}'],
    'Avg justification tokens': [f'{df_j["just_tokens"].mean():.1f}'],
    'Median justification tokens': [f'{df_j["just_tokens"].median():.0f}'],
    'Avg just/claim ratio'    : [f'{(df_j["just_tokens"]/df_j["claim_tokens"]).mean():.1f}×'],
}).T.rename(columns={0:'Value'})

from IPython.display import display
display(summary)


## 11. Written Analysis

### 11.1 Label Distribution

The LIAR-PLUS dataset is distributed across six veracity labels with a **moderate imbalance**: 
`half-true` is the most frequent class (~21%), followed by `false` (~18%) and `mostly-true` (~18%), 
while `pants-fire` is the rarest (~8%). This skew is consistent across splits, confirming that 
Alhindi et al. preserved the original LIAR proportions. For the classifier, this imbalance is modest 
enough to be handled with standard class weights or oversampling, but it is worth noting.

### 11.2 Claim Text

Claims are short, focused statements. The median claim length is approximately **18–20 tokens** 
with a long right tail reaching 80+ tokens. The distribution is right-skewed: most claims are 
concise political assertions, while a smaller number consist of extended quotes or multi-clause 
sentences. Structurally, the majority of claims contain a **proper noun** (speaker name, politician, 
institution), roughly 60–70% include a **number or statistic**, and about 20–30% include 
**direct quotation marks** or an attribution phrase. Temporal references appear in about half 
of all claims. Notably, `pants-fire` claims tend to be **slightly shorter** than `true` or 
`mostly-true` claims, suggesting that the most egregious falsehoods are often bolder, simpler 
assertions.

### 11.3 Justification Text

Justifications are substantially longer than claims, with a median around **65–80 tokens** 
and a mean above 90 tokens (ratio ≈ 4–5× the corresponding claim). The distribution is again 
right-skewed: simple verdicts may be rendered in 20 tokens, while complex fact-checks can 
exceed 200 tokens. A key finding is that justifications are **far from formulaic**: the most 
common opening phrases ("the claim is", "there is no", "according to") account for only a 
small fraction of all instances, suggesting that the original human justifications are 
stylistically varied. This heterogeneity is relevant for synthetic generation: the model 
should not be constrained to a rigid output template.

### 11.4 Speaker, Party and Subject

Democrats and Republicans together account for the majority of statements (~85%), with Democrats 
contributing slightly more statements but Republicans showing a higher proportion of 
`pants-fire` and `false` labels. The top speakers — Barack Obama, Donald Trump, Hillary Clinton — 
each contribute hundreds of statements, introducing potential **speaker-level contamination risk**: 
models trained extensively on text about these figures are likely to have learned associations 
between their names, topics, and typical rhetorical patterns. The most frequent subjects are 
`health-care`, `economy`, `taxes`, `education`, and `elections` — a set that heavily overlaps 
with the general political discourse covered in LLM pre-training corpora, reinforcing the 
data leakage concerns outlined in Chapter 4.2.

### 11.5 Speaker Credit History

The credit history columns (barely-true, false, half-true, mostly-true, pants-fire counts) 
encode a speaker's **lifetime veracity profile** at the time of each statement. The distributions 
are strongly right-skewed: most speakers in the dataset have made fewer than 20 statements 
previously, but high-volume political figures drive the tail. There is a mild trend suggesting 
that speakers with longer track records are not systematically more or less truthful, but 
experienced speakers (50+ past statements) show a slightly lower `pants-fire` rate, consistent 
with the idea that LIAR skews toward established political figures for its most prolific speakers.

### 11.6 Implications for Input Engineering

Several findings from this analysis have direct implications for the prompt design phase:

1. **Claim length is short and controllable.** With a median of ~18 tokens, full claims can be 
   included verbatim in any prompt without context window concerns, even for batch generation.

2. **Debranding targets are clearly identifiable.** The high prevalence of proper nouns, 
   attributions, and temporal references means that a systematic debranding procedure 
   (replacing named entities, normalising dates) is feasible via regex or NER and will cover 
   the vast majority of contamination risk patterns identified in Chapter 4.2.

3. **Justification length should be guided, not fixed.** Given the 4–5× ratio and high variance, 
   a prompt instruction specifying a word-count range (e.g., 50–120 tokens) may help reduce 
   justification length variance across models without over-constraining the output.

4. **Topic and speaker metadata are available and informative.** Including the subject tag 
   and speaker job title in the prompt could help the model contextualise the claim appropriately 
   without introducing direct label information — consistent with blind generation.

5. **Multi-label structure should be preserved.** Since the dataset uses six fine-grained labels 
   rather than binary true/false, the evaluation framework and any label-dependent analysis 
   must account for this ordering. The EDA confirms the six-class distribution is stable 
   across splits, supporting consistent cross-split comparisons.
